# SHAP -- Logistic Regression (Sex-Specific: Male + Female)

Repeated 5-fold cross-validation, 10 seeds (50 folds total), across all 5 prediction horizons (1-5 years), computed separately for male-only and female-only populations.

**Output**: mean |SHAP| and standard deviation per feature, averaged across the 50 folds, for each sex.

In [1]:
import ast
import numpy as np
import pandas as pd
import shap
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier, ExtraTreesClassifier,
                               BaggingClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
SEEDS = list(range(10))
HORIZONS = [1, 2, 3, 4, 5]
N_BACKGROUND = 20
MAX_EVAL = 40

pd.set_option('display.max_columns', None)

## Model config

In [2]:
MODEL_CONFIGS = {
    'Logistic Regression': dict(cls=LogisticRegression, fixed_kwargs=dict(solver='saga', max_iter=5000), scaled=True, has_seed=True),
}
MODEL_NAME = 'Logistic Regression'
PREFIX = 'logreg'
EXPLAINER_KIND = 'linear'
SEXES = ['M', 'F']

## Run: 50-fold repeated CV + SHAP per fold, all 5 horizons x each sex

In [3]:
def make_preprocessor(scaled, numeric_cols, comorbid_cols):
    num_step = StandardScaler() if scaled else 'passthrough'
    return ColumnTransformer([('num', num_step, numeric_cols), ('bin', 'passthrough', comorbid_cols)])

def load_best_params(sex, h):
    results = pd.read_csv('sex_specific_results.csv')
    row = results[(results['model'] == MODEL_NAME) & (results['sex'] == sex) & (results['horizon'] == f'{h}y')].iloc[0]
    params = ast.literal_eval(row['best_params'])
    return {k.replace('clf__', ''): v for k, v in params.items()}

df = pd.read_csv('modeling_dataset.csv')
COMORBID_COLS = [c for c in df.columns if c not in (
    ['person_id', 'sex', 'age', 'postcode', 'rurality', 'ses_irsd_decile', 'ses_missing',
     'incident_cvd', 'years_followup', 'split',
     'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
     'baseline_n_episodes', 'baseline_history_days']
    + [f'label_{hh}y' for hh in range(1, 6)] + [f'eligible_{hh}y' for hh in range(1, 6)]
)]
NUMERIC_COLS = ['age', 'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                'baseline_n_episodes', 'baseline_history_days']
FEATURE_COLS = NUMERIC_COLS + COMORBID_COLS  # sex excluded -- constant within each sex-specific subset

cfg = MODEL_CONFIGS[MODEL_NAME]

for sex in SEXES:
    all_results = []
    for h in HORIZONS:
        preprocessor = make_preprocessor(cfg['scaled'], NUMERIC_COLS, COMORBID_COLS)
        base_params = load_best_params(sex, h)
        base_params.update(cfg['fixed_kwargs'])

        elig_col, label_col = f'eligible_{h}y', f'label_{h}y'
        sub = df[(df[elig_col]) & (df['sex'] == sex)].copy()
        X_all, y_all = sub[FEATURE_COLS], sub[label_col].astype(int)

        fold_importances = []
        n_folds_done = 0

        for seed in SEEDS:
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for train_idx, test_idx in skf.split(X_all, y_all):
                X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
                y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
                if y_test.nunique() < 2 or y_train.nunique() < 2:
                    continue

                run_params = dict(base_params)
                if cfg.get('needs_scale_pos_weight'):
                    run_params['scale_pos_weight'] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                if cfg['has_seed']:
                    run_params['random_state'] = seed

                clf_obj = cfg['cls'](**cfg.get('base_kwargs', {}))
                pipe = Pipeline([('preprocess', preprocessor), ('clf', clf_obj)])
                pipe.set_params(**{f'clf__{k}': v for k, v in run_params.items()})
                pipe.fit(X_train, y_train)

                fit_preprocessor, clf = pipe.named_steps['preprocess'], pipe.named_steps['clf']
                Xt_test = fit_preprocessor.transform(X_test)
                feature_names = fit_preprocessor.get_feature_names_out()

                try:
                    if EXPLAINER_KIND == 'tree':
                        explainer = shap.TreeExplainer(clf)
                        sv = explainer.shap_values(Xt_test)
                        if isinstance(sv, list):
                            sv = sv[1]
                        elif sv.ndim == 3:
                            sv = sv[:, :, 1]
                    elif EXPLAINER_KIND == 'linear':
                        Xt_train = fit_preprocessor.transform(X_train)
                        explainer = shap.LinearExplainer(clf, Xt_train)
                        sv = explainer.shap_values(Xt_test)
                    else:
                        Xt_train = fit_preprocessor.transform(X_train)
                        background = shap.sample(Xt_train, min(N_BACKGROUND, len(Xt_train)), random_state=RANDOM_STATE)
                        eval_idx = np.random.RandomState(RANDOM_STATE).choice(len(Xt_test), min(MAX_EVAL, len(Xt_test)), replace=False)
                        Xt_eval = Xt_test[eval_idx] if not hasattr(Xt_test, 'iloc') else Xt_test.iloc[eval_idx]
                        explainer = shap.KernelExplainer(lambda x: clf.predict_proba(x)[:, 1], background)
                        sv = explainer.shap_values(Xt_eval, silent=True)

                    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=feature_names)
                    fold_importances.append(mean_abs)
                    n_folds_done += 1
                except Exception as e:
                    print(f'  SHAP failed on one fold ({MODEL_NAME}/{sex}/{h}y): {e}')
                    continue

        imp_df = pd.DataFrame(fold_importances)
        mean_imp = imp_df.mean(axis=0).sort_values(ascending=False)
        std_imp = imp_df.std(axis=0)
        top5_per_fold = imp_df.apply(lambda row: set(row.sort_values(ascending=False).head(5).index), axis=1)
        top10_features = mean_imp.head(10).index

        for feat in top10_features:
            stability = sum(feat in t5 for t5 in top5_per_fold) if len(top5_per_fold) else 0
            all_results.append(dict(model=MODEL_NAME, sex=sex, horizon=f'{h}y', feature=feat,
                                     mean_abs_shap=mean_imp[feat], std_abs_shap=std_imp.get(feat, np.nan),
                                     top5_stability=f'{stability}/{n_folds_done}'))

        print(f'{MODEL_NAME} / {sex} / {h}y: done ({n_folds_done} folds), top feature = {mean_imp.index[0]}')

    results_df = pd.DataFrame(all_results)
    OUT_FILE = f'sexspecific_shap_repeated_cv_{PREFIX}_{sex}.csv'
    results_df.to_csv(OUT_FILE, index=False)
    print('Saved', OUT_FILE)

Background dataset has 1201 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1201 when initializing the masker.


Background dataset has 1201 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1201 when initializing the masker.


Background dataset has 1202 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1202 when initializing the masker.


Background dataset has 1202 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1202 when initializing the masker.


Background dataset has 1202 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1202 when initializing the masker.


...

Background dataset has 310 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=310 when initializing the masker.


Background dataset has 310 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=310 when initializing the masker.


Background dataset has 311 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=311 when initializing the masker.


Background dataset has 311 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=311 when initializing the masker.


Logistic Regression / F / 5y: done (50 folds), top feature = num__baseline_n_episodes
Saved sexspecific_shap_repeated_cv_logreg_F.csv
